## 1(a)

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 5)

DATA_DIR = Path('data') / 'AReM'
if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        'Run this notebook from the folder containing data/AReM.'
    )

channels = [
    'avg_rss12', 'var_rss12', 'avg_rss13',
    'var_rss13', 'avg_rss23', 'var_rss23',
]
activities = [
    'bending1', 'bending2', 'cycling', 'lying',
    'sitting', 'standing', 'walking',
]

records, cleaning = [], []
for activity in activities:
    files = sorted(
        (DATA_DIR / activity).glob('dataset*.csv'),
        key=lambda p: int(re.search(r'dataset(\d+)', p.stem).group(1)),
    )
    for file in files:
        name = file.relative_to(DATA_DIR).as_posix()
        rows, whitespace_rows, trailing_rows = [], 0, 0
        for line in file.read_text(encoding='utf-8-sig').splitlines():
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            whitespace_rows += int(',' not in line)
            trailing_rows += int(line.endswith(','))
            fields = re.split(r'[,\s]+', line.rstrip(',').strip())
            if len(fields) != 7:
                raise ValueError((name, 'Expected seven fields', line))
            rows.append([float(x) for x in fields])

        array = np.asarray(rows)
        if name == 'sitting/dataset8.csv' and array.shape == (479, 7):
            grid = np.arange(0, 120000, 250)
            missing = sorted(set(grid) - set(array[:, 0]))
            assert missing == [13500], missing
            interpolated = [
                np.interp(grid, array[:, 0], array[:, j])
                for j in range(1, 7)
            ]
            array = np.column_stack([grid] + interpolated)
            cleaning.append({
                'file': name,
                'repair': 'Interpolated six values at t=13500',
            })

        assert array.shape == (480, 7), (name, array.shape)
        assert np.isfinite(array).all(), name
        assert (np.diff(array[:, 0]) > 0).all(), name
        number = int(re.search(r'dataset(\d+)', file.stem).group(1))
        records.append({
            'activity': activity,
            'file': name,
            'number': number,
            'values': array[:, 1:],
        })
        if whitespace_rows or trailing_rows:
            cleaning.append({
                'file': name,
                'whitespace_rows': whitespace_rows,
                'trailing_comma_rows': trailing_rows,
            })

assert len(records) == 88
print('Verified: 88 samples, each 480 x 6; all values finite.')
display(pd.DataFrame(cleaning))


Verified: 88 samples, each 480 x 6; all values finite.


,file,whitespace_rows,trailing_comma_rows,repair
0,bending2/dataset4.csv,480.0,0.0,NaN
1,cycling/dataset9.csv,0.0,1.0,NaN
2,cycling/dataset14.csv,0.0,1.0,NaN
3,sitting/dataset8.csv,NaN,NaN,Interpolated six values at t=13500


## 1(b)

In [2]:
metadata = pd.DataFrame(
    [
        {
            k: r[
                k] for k in [
                'activity',
                'file',
                'number']} for r in records])
metadata.insert(0, 'Instance', np.arange(1, 89))
metadata['split'] = [
    'test'
    if r['number'] <= (
        2 if r['activity'].startswith('bending') else 3
    )
    else 'train'
    for r in records
]
split_counts = pd.crosstab(
    metadata[
        'activity'],
    metadata[
        'split']).reindex(
    activities)
display(split_counts.assign(total=split_counts.sum(axis=1)))
assert metadata['split'].value_counts().to_dict() == {'train': 69, 'test': 19}
print('Test files:')
display(
    metadata.loc[
        metadata[
            'split'].eq(
            'test'),
        [
            'activity',
            'file']].reset_index(
        drop=True))


split,test,train,total
activity,,,
bending1,2,5,7
bending2,2,4,6
cycling,3,12,15
lying,3,12,15
sitting,3,12,15
standing,3,12,15
walking,3,12,15


Test files:


,activity,file
0,bending1,bending1/dataset1.csv
1,bending1,bending1/dataset2.csv
2,bending2,bending2/dataset1.csv
3,bending2,bending2/dataset2.csv
4,cycling,cycling/dataset1.csv
5,cycling,cycling/dataset2.csv
6,cycling,cycling/dataset3.csv
7,lying,lying/dataset1.csv
8,lying,lying/dataset2.csv
9,lying,lying/dataset3.csv


## 1(c)(i)

In [3]:
feature_types = pd.DataFrame(
    [
        ("Location", "Mean, median, quantiles"),
        ("Extremes", "Minimum, maximum, range"),
        ("Dispersion", "Variance, standard deviation, IQR, MAD"),
        ("Shape", "Skewness, kurtosis"),
        ("Amplitude", "RMS, sum of squared values"),
        ("Changes", "Mean absolute difference, zero-crossing rate"),
        ("Temporal structure", "Autocorrelation, trend, peak count"),
    ],
    columns=["Category", "Examples"],
)
display(feature_types)
print(
    "Reference: "
    "https://tsfresh.readthedocs.io/en/latest/text/list_of_features.html"
)


,Category,Examples
0,Location,"Mean, median, quantiles"
1,Extremes,"Minimum, maximum, range"
2,Dispersion,"Variance, standard deviation, IQR, MAD"
3,Shape,"Skewness, kurtosis"
4,Amplitude,"RMS, sum of squared values"
5,Changes,"Mean absolute difference, zero-crossing rate"
6,Temporal structure,"Autocorrelation, trend, peak count"


Reference: https://tsfresh.readthedocs.io/en/latest/text/list_of_features.html


## 1(c)(ii)

In [4]:
feature_rows = []
for record in records:
    row = {}
    for j, x in enumerate(record['values'].T, start=1):
        stats = {'min': np.min(x), 'max': np.max(x), 'mean': np.mean(x),
                 'median': np.median(x), 'std': np.std(x, ddof=1),
                 'q1_': np.quantile(x, 0.25, method='linear'),
                 'q3_': np.quantile(x, 0.75, method='linear')}
        row.update({f'{name}{j}': value for name, value in stats.items()})
    feature_rows.append(row)
features = pd.DataFrame(
    feature_rows,
    index=pd.Index(
        metadata[
            'Instance'],
        name='Instance'))
assert features.shape == (88, 42)
assert np.isfinite(features.to_numpy()).all()
train_mask = metadata['split'].eq('train').to_numpy()
X_train, X_test = features.loc[
    train_mask].copy(
    ), features.loc[
    ~train_mask].copy(
    )
y_train, y_test = metadata.loc[
    train_mask,
    'activity'].copy(
    ), metadata.loc[
    ~train_mask,
    'activity'].copy(
    )
print(
    'Feature matrix:',
    features.shape,
    'Train:',
    X_train.shape,
    'Test:',
    X_test.shape)
with pd.option_context('display.max_rows', 100, 'display.max_columns', 50):
    display(features)


Feature matrix: (88, 42) Train: (69, 42) Test: (19, 42)


,min1,max1,mean1,median1,std1,q1_1,q3_1,min2,max2,mean2,median2,std2,q1_2,q3_2,min3,max3,mean3,median3,std3,q1_3,q3_3,min4,max4,mean4,median4,std4,q1_4,q3_4,min5,max5,mean5,median5,std5,q1_5,q3_5,min6,max6,mean6,median6,std6,q1_6,q3_6
Instance,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.62479,40.500,1.47697,39.2500,42.0000,0.0,1.30,0.35860,0.430,0.32260,0.0000,0.5000,4.00,29.50,19.04094,19.250,4.46295,16.0000,23.2500,0.0,7.23,0.83254,0.500,0.96566,0.0000,1.1200,27.25,38.25,34.31129,35.000,2.18845,33.0000,36.0000,0.00,1.92,0.57058,0.430,0.58292,0.0000,1.3000
2,38.00,45.67,42.81281,42.500,1.43555,42.0000,43.6700,0.0,1.22,0.37244,0.470,0.28916,0.0000,0.5000,2.00,29.50,20.09698,21.000,3.89374,19.0000,22.2500,0.0,5.76,0.87644,0.500,1.04726,0.0000,1.1450,27.67,38.50,33.02458,33.000,1.99525,32.0000,34.5000,0.00,3.11,0.57108,0.430,0.60101,0.0000,1.3000
3,35.00,47.40,43.95450,44.330,1.55884,43.0000,45.0000,0.0,1.70,0.42625,0.470,0.33869,0.0000,0.5000,6.50,29.75,22.12235,23.000,3.03094,19.7500,24.0000,0.0,4.44,0.49731,0.430,0.55066,0.0000,0.8300,29.00,38.50,35.58846,36.000,1.99960,35.3625,36.5000,0.00,1.79,0.49329,0.430,0.51351,0.0000,0.9400
4,33.00,47.75,42.17981,43.500,3.67067,39.1500,45.0000,0.0,3.00,0.69604,0.500,0.63086,0.0000,1.1200,8.50,30.00,22.18362,23.000,3.81047,20.5000,24.3725,0.0,5.15,0.98992,0.830,0.95373,0.4300,1.3000,20.00,38.67,33.49392,35.000,3.84945,30.4575,36.3300,0.00,2.18,0.61352,0.500,0.52432,0.0000,1.0000
5,33.00,45.75,41.67806,41.750,2.24349,41.3300,42.7500,0.0,2.83,0.53598,0.500,0.40547,0.4300,0.7100,3.00,28.25,19.00656,19.125,4.08711,16.5000,22.0625,0.0,6.42,0.84188,0.500,0.92880,0.4300,1.1200,23.67,37.50,29.85708,30.000,2.41103,28.4575,31.2500,0.00,1.79,0.38329,0.430,0.38916,0.0000,0.5000
6,37.00,48.00,43.45496,43.250,1.38610,42.5000,45.0000,0.0,1.58,0.37808,0.470,0.31557,0.0000,0.5000,5.75,27.00,15.79333,15.000,3.84764,13.0000,18.2700,0.0,10.03,0.84935,0.500,0.99576,0.4300,1.1200,8.00,33.50,23.03479,23.500,2.48886,22.2500,24.0000,0.00,5.26,0.67965,0.500,0.62253,0.4300,0.8700
7,36.25,48.00,43.96912,44.500,1.61836,43.3100,44.6700,0.0,1.50,0.41312,0.470,0.26311,0.4300,0.5000,1.50,26.33,15.86802,16.250,3.74242,14.2500,18.0000,0.0,5.17,0.66635,0.470,0.78898,0.0000,0.9400,11.33,30.75,22.10375,21.670,3.31830,20.5000,23.7500,0.00,2.96,0.55531,0.490,0.48783,0.0000,0.8300
8,12.75,51.00,24.56296,24.250,3.73751,23.1875,26.5000,0.0,6.87,0.59083,0.430,0.83741,0.0000,0.7100,0.00,25.33,19.12133,20.250,4.29961,18.0000,22.0000,0.0,6.76,0.73677,0.470,0.92705,0.0000,0.8700,10.67,30.00,23.49304,23.750,3.69379,20.5000,27.0000,0.00,4.97,0.70019,0.500,0.69372,0.4300,0.8700
9,0.00,42.75,27.46460,28.000,3.58358,25.5000,30.0000,0.0,7.76,0.44971,0.430,0.76720,0.0000,0.5000,7.50,35.00,20.84254,20.750,3.82627,18.0000,22.5000,0.0,5.76,0.77992,0.500,0.90444,0.0000,0.9400,0.00,33.00,17.61794,18.000,5.05364,15.0000,20.7500,0.00,6.76,1.12213,0.830,1.01234,0.4700,1.3000


## 1(c)(iii)

In [5]:
SEED, B, BATCH = 552, 20000, 250
rng = np.random.default_rng(SEED)
A = features.to_numpy()
n, p = A.shape
bootstrap_sd = np.empty((B, p))
for start in range(0, B, BATCH):
    stop = min(start + BATCH, B)
    indices = rng.integers(0, n, size=(stop - start, n))
    bootstrap_sd[start:stop] = A[indices].std(axis=1, ddof=1)
lower, upper = np.quantile(bootstrap_sd, [0.05, 0.95], axis=0)
ci_table = pd.DataFrame(
    {
        'feature': features.columns,
        'sample_std': A.std(
            axis=0,
            ddof=1),
                         'CI_90_lower': lower, 'CI_90_upper': upper})
assert ci_table.shape == (42, 4)
assert (lower <= upper).all()
with pd.option_context('display.max_rows', 50):
    display(ci_table)


,feature,sample_std,CI_90_lower,CI_90_upper
0,min1,9.56998,8.27027,10.75838
1,max1,4.39436,3.32187,5.29486
2,mean1,5.33570,4.70790,5.88112
3,median1,5.44005,4.79378,5.99609
4,std1,1.77216,1.57133,1.95161
5,q1_1,6.15359,5.58210,6.64543
6,q3_1,5.13892,4.33194,5.84072
7,min2,0.00000,0.00000,0.00000
8,max2,5.06273,4.62504,5.40704
9,mean2,1.57417,1.39410,1.70688


## 1(c)(iv)

In [6]:
selected_columns = [
    f'{
        stat}{
        j}' for j in range(
        1,
        7) for stat in [
        'mean',
        'std',
        'median']]
selected_train = X_train[selected_columns]
selected_test = X_test[selected_columns]
print('Selected 18 features:', ', '.join(selected_columns))
print(
    'Training shape:',
    selected_train.shape,
    'Test shape:',
    selected_test.shape)
display(selected_train.head())

selection_reasons = pd.DataFrame(
    [
        ("Mean", "Typical signal level"),
        ("Standard deviation", "Variation over time"),
        ("Median", "Robust central level; less sensitive to outliers"),
    ],
    columns=["Feature", "Rationale"],
)
display(selection_reasons)
print(
    "This is a judgment-based choice, not a validated optimal subset. "
    "Mean and median can be correlated. "
    "Use training-only cross-validation for further selection."
)


Selected 18 features: mean1, std1, median1, mean2, std2, median2, mean3, std3, median3, mean4, std4, median4, mean5, std5, median5, mean6, std6, median6
Training shape: (69, 18) Test shape: (19, 18)


,mean1,std1,median1,mean2,std2,median2,mean3,std3,median3,mean4,std4,median4,mean5,std5,median5,mean6,std6,median6
Instance,,,,,,,,,,,,,,,,,,
3,43.95450,1.55884,44.33,0.42625,0.33869,0.47,22.12235,3.03094,23.000,0.49731,0.55066,0.43,35.58846,1.99960,36.00,0.49329,0.51351,0.43
4,42.17981,3.67067,43.50,0.69604,0.63086,0.50,22.18362,3.81047,23.000,0.98992,0.95373,0.83,33.49392,3.84945,35.00,0.61352,0.52432,0.50
5,41.67806,2.24349,41.75,0.53598,0.40547,0.50,19.00656,4.08711,19.125,0.84188,0.92880,0.50,29.85708,2.41103,30.00,0.38329,0.38916,0.43
6,43.45496,1.38610,43.25,0.37808,0.31557,0.47,15.79333,3.84764,15.000,0.84935,0.99576,0.50,23.03479,2.48886,23.50,0.67965,0.62253,0.50
7,43.96912,1.61836,44.50,0.41312,0.26311,0.47,15.86802,3.74242,16.250,0.66635,0.78898,0.47,22.10375,3.31830,21.67,0.55531,0.48783,0.49


,Feature,Rationale
0,Mean,Typical signal level
1,Standard deviation,Variation over time
2,Median,Robust central level; less sensitive to outliers


This is a judgment-based choice, not a validated optimal subset. Mean and median can be correlated. Use training-only cross-validation for further selection.
